In [1]:
import pandas as pd
import numpy as np

In [9]:
import pandas as pd

df = pd.read_csv("Downloads/survey_data_updated 5.xls")
print("Shape of raw dataset:", df.shape)
df.head()

Shape of raw dataset: (18845, 114)


,ResponseId,MainBranch,Age,Employment,RemoteWork,Check,CodingActivities,EdLevel,LearnCode,LearnCodeOnline,...,JobSatPoints_6,JobSatPoints_7,JobSatPoints_8,JobSatPoints_9,JobSatPoints_10,JobSatPoints_11,SurveyLength,SurveyEase,ConvertedCompYearly,JobSat
0,2,I am a developer by profession,35-44 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,"Bachelor's degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
1,3,I am a developer by profession,45-54 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects;Other...,"Master's degree (M.A., M.S., M.Eng., MBA, etc.)",Books / Physical media;Colleague;On the job tr...,Technical documentation;Blogs;Books;Written Tu...,...,NaN,NaN,NaN,NaN,NaN,NaN,Appropriate in length,Easy,NaN,NaN
2,10,I am a developer by profession,35-44 years old,"Independent contractor, freelancer, or self-em...",Remote,Apples,Bootstrapping a business,"Master's degree (M.A., M.S., M.Eng., MBA, etc.)",On the job training;Other online resources (e....,Technical documentation;Blogs;Written Tutorial...,...,NaN,NaN,NaN,NaN,NaN,NaN,Too long,Easy,NaN,NaN
3,11,"I used to be a developer by profession, but no...",35-44 years old,"Employed, full-time",Remote,Apples,Hobby;Contribute to open-source projects,"Bachelor's degree (B.A., B.S., B.Eng., etc.)",Books / Physical media;Other online resources ...,Technical documentation;Books;Written Tutorial...,...,25.0,10.0,0.0,15.0,0.0,0.0,Appropriate in length,Easy,NaN,8.0
4,12,I am a developer by profession,45-54 years old,"Employed, full-time",In-person,Apples,Hobby;School or academic work,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)","Books / Physical media;School (i.e., Universit...",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Appropriate in length,Neither easy nor difficult,NaN,NaN


In [11]:
df.to_csv("Downloads/survey_data_raw.csv", index=False)
print("Saved correctly as CSV")

Saved correctly as CSV


In [12]:
# ==========================================
# Convert multi-value columns into rows
# ==========================================

def explode_column(df, column, tech_type, category):
    
    # Select relevant columns
    temp = df[[column, "Age", "Country", "EdLevel"]].dropna()
    
    # Split values (they are separated by ;)
    temp[column] = temp[column].str.split(";")
    
    # Convert list into multiple rows
    temp = temp.explode(column)
    
    # Clean spaces
    temp[column] = temp[column].str.strip()
    
    # Rename column to match final dataset
    temp = temp.rename(columns={column: "TechValue"})
    
    # Add new columns (same as your dataset)
    temp["RecordType"] = "Technology"
    temp["TechCategory"] = category
    temp["TechType"] = tech_type
    
    return temp[["Age", "Country", "EdLevel", "RecordType", "TechCategory", "TechType", "TechValue"]]

In [13]:
# ==========================================
#  Create technology records
# ==========================================

tech_data = []

mapping = [
    ("LanguageHaveWorkedWith", "Language", "Worked"),
    ("DatabaseHaveWorkedWith", "Database", "Worked"),
    ("PlatformHaveWorkedWith", "Platform", "Worked"),
    ("WebFrameHaveWorkedWith", "WebFrame", "Worked"),
    ("LanguageWantToWorkWith", "Language", "Want"),
    ("DatabaseWantToWorkWith", "Database", "Want"),
    ("PlatformWantToWorkWith", "Platform", "Want"),
    ("WebFrameWantToWorkWith", "WebFrame", "Want")
]

for col, tech_type, category in mapping:
    if col in df.columns:
        tech_data.append(explode_column(df, col, tech_type, category))

technology_df = pd.concat(tech_data, ignore_index=True)

print("Technology dataset shape:", technology_df.shape)
technology_df.head()

Technology dataset shape: (457477, 7)


,Age,Country,EdLevel,RecordType,TechCategory,TechType,TechValue
0,35-44 years old,United Kingdom of Great Britain and Northern I...,"Bachelor's degree (B.A., B.S., B.Eng., etc.)",Technology,Worked,Language,Bash/Shell (all shells)
1,35-44 years old,United Kingdom of Great Britain and Northern I...,"Bachelor's degree (B.A., B.S., B.Eng., etc.)",Technology,Worked,Language,Go
2,35-44 years old,United Kingdom of Great Britain and Northern I...,"Bachelor's degree (B.A., B.S., B.Eng., etc.)",Technology,Worked,Language,HTML/CSS
3,35-44 years old,United Kingdom of Great Britain and Northern I...,"Bachelor's degree (B.A., B.S., B.Eng., etc.)",Technology,Worked,Language,Java
4,35-44 years old,United Kingdom of Great Britain and Northern I...,"Bachelor's degree (B.A., B.S., B.Eng., etc.)",Technology,Worked,Language,JavaScript


In [16]:
# ==========================================
# Create demographics records
# ==========================================

demo_df = df[["ResponseId", "Age", "Country", "EdLevel"]].copy()

demo_df["RecordType"] = "Demographics"
demo_df["TechCategory"] = None
demo_df["TechType"] = None
demo_df["TechValue"] = None

demo_df = demo_df[[
    "ResponseId", "Age", "Country", "EdLevel",
    "RecordType", "TechCategory", "TechType", "TechValue"
]]

print("Demographics dataset shape:", demo_df.shape)
demo_df.head()

Demographics dataset shape: (18845, 8)


,ResponseId,Age,Country,EdLevel,RecordType,TechCategory,TechType,TechValue
0,2,35-44 years old,United Kingdom of Great Britain and Northern I...,"Bachelor's degree (B.A., B.S., B.Eng., etc.)",Demographics,None,None,None
1,3,45-54 years old,United Kingdom of Great Britain and Northern I...,"Master's degree (M.A., M.S., M.Eng., MBA, etc.)",Demographics,None,None,None
2,10,35-44 years old,Serbia,"Master's degree (M.A., M.S., M.Eng., MBA, etc.)",Demographics,None,None,None
3,11,35-44 years old,United States of America,"Bachelor's degree (B.A., B.S., B.Eng., etc.)",Demographics,None,None,None
4,12,45-54 years old,Poland,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)",Demographics,None,None,None


In [17]:
# ==========================================
# Combine final dataset
# ==========================================

technology_df = technology_df.merge(
    df[["ResponseId", "Age", "Country", "EdLevel"]],
    on=["Age", "Country", "EdLevel"],
    how="left"
)

final_df = pd.concat([demo_df, technology_df], ignore_index=True)

print("Final dataset shape:", final_df.shape)
final_df.head()

Final dataset shape: (54637759, 8)


,ResponseId,Age,Country,EdLevel,RecordType,TechCategory,TechType,TechValue
0,2,35-44 years old,United Kingdom of Great Britain and Northern I...,"Bachelor's degree (B.A., B.S., B.Eng., etc.)",Demographics,None,None,None
1,3,45-54 years old,United Kingdom of Great Britain and Northern I...,"Master's degree (M.A., M.S., M.Eng., MBA, etc.)",Demographics,None,None,None
2,10,35-44 years old,Serbia,"Master's degree (M.A., M.S., M.Eng., MBA, etc.)",Demographics,None,None,None
3,11,35-44 years old,United States of America,"Bachelor's degree (B.A., B.S., B.Eng., etc.)",Demographics,None,None,None
4,12,45-54 years old,Poland,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)",Demographics,None,None,None


In [18]:
# ==========================================
# Save final dashboard dataset
# ==========================================

final_df.to_csv("Downloads/final_dashboard_dataset.csv", index=False)
print("Final dashboard dataset saved successfully")

Final dashboard dataset saved successfully
